# Explorando os dados de logs do serviço de Streaming

Temos três arquivos disponíveis para analisar na temática de um serviço de streaming:

- `catalogo_filmes.csv`: dataset registrando o acervo da plataforma
- `usuarios.csv`: dataset com registros dos usuários da plataforma
- `logs_streaming.csv`: principal dataset, possui registros de sessões de streaming dos usuários

Vamos começar lendo o conteúdo de `logs_streaming.csv` para um dataframe Spark e depois vamos checar o schema que o Spark inferiu para as colunas:

In [0]:
df_logs = spark.read.csv(
    "/Volumes/academy/dados-streaing-filmes/logs_streaming.csv",
    header=True, # primeira linha do arquivo eh o cabecalho
    inferSchema=True # infere o schema do dataframe
)

# shape do dataframe
print((df_logs.count(), len(df_logs.columns)))

# display(df_logs) # mostra todas as linhas
df_logs.limit(10).display() # mostra apenas as 10 primeiras

(1007, 10)


session_id,user_id,movie_id,device,country,subscription_type,watch_time_minutes,watch_date,rating,playback_status
SFCF6C0D030,U0064,M0016,null,Colômbia,standard,179,05-18-2026,1,erro
S228044CE8C,U0036,M0024,Smart TV,Espanha,free,26,25/02/2026,2,paused
S2A082D3FBC,U0059,M0014,web,BR,premium,33,2026/01/06,2,COMPLETED
SF47C1318E3,U0077,M0055,smart_tv,Chile,free,29,2026/02/07,2,COMPLETED
S74DAA8499B,U0034,M0048,null,Argentina,null,93,2026-04-18,5,completed
SEC8BF1659B,U0067,M0042,null,null,basic,28,01-08-2026,3,buffering
S69074240D4,U0036,M0038,console,null,trial,136,05-21-2026,2,null
S441D657A8B,U0028,M0057,Smart TV,Colômbia,BASIC,25,02-28-2026,3,buffering
SF2F56598CE,U0061,M0054,tablet,Brazil,BASIC,96,2026-01-22,4,COMPLETED
S30835F96D2,U0055,M0047,null,México,standard,148,2026/01/14,2,paused


In [0]:
df_logs.printSchema()

root
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- movie_id: string (nullable = true)
 |-- device: string (nullable = true)
 |-- country: string (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- watch_time_minutes: string (nullable = true)
 |-- watch_date: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- playback_status: string (nullable = true)



## 1 Campos de IDs na base de logs

A chave primária da base de logs é a coluna `session_id`, apenas ela já deveria ser suficiente para ser usada como PK. Mas como temos outros campos de IDs, e que serão importantes para relacionar com as outras bases, vamos ver o que há neles também.

Farei isso usando comandos SQL:

In [0]:
# cria tabela temporaria para ser acessada por SQL
df_logs.createOrReplaceTempView('logs_streaming')

In [0]:
%sql
-- procura por valores duplicados nas colunas de IDs
select count(distinct session_id) as unique_session_id
    ,count(distinct user_id) as unique_user_id
    ,count(distinct movie_id) as unique_movie_id
    ,count(distinct session_id, user_id, movie_id) as row_id
    ,count(*) as total_linhas
from logs_streaming

unique_session_id,unique_user_id,unique_movie_id,row_id,total_linhas
1001,80,60,997,1007


In [0]:
%sql
-- linhas com session_id duplicado
select a.*
from logs_streaming a
    inner join (
        select session_id from logs_streaming group by session_id having count(*) > 1
    ) b on a.session_id = b.session_id
order by a.session_id

session_id,user_id,movie_id,device,country,subscription_type,watch_time_minutes,watch_date,rating,playback_status
S24C425C86E,U0002,M0002,tablet,Portugal,Premium,11,05-15-2026,null,error
S24C425C86E,U0002,M0002,tablet,Portugal,Premium,11,05-15-2026,null,error
S726D62D25A,U0039,M0046,tablet,United States,standard,174,2026/01/30,3,null
S726D62D25A,U0039,M0046,tablet,United States,standard,174,2026/01/30,3,null
S8819FA5D1C,U0072,M0049,null,Brasil,Premium,160,03-13-2026,5,cancelled
S8819FA5D1C,U0072,M0049,null,Brasil,Premium,160,03-13-2026,5,cancelled
SBB00A2D022,U0053,M0021,Smart TV,Portugal,premium,176,26/04/2026,4.5,buffering
SBB00A2D022,U0053,M0021,Smart TV,Portugal,premium,176,26/04/2026,4.5,buffering
SDB9756D3FE,U0020,M0054,web,null,Premium,174,2026-02-12,4,error
SDB9756D3FE,U0020,M0054,web,null,Premium,174,2026-02-12,4,error


In [0]:
df_logs = df_logs.dropDuplicates(subset=['session_id', 'user_id', 'movie_id'])

# shape do dataframe
print((df_logs.count(), len(df_logs.columns)))

(1001, 10)


In [0]:
# (re)cria tabela temporaria para ser acessada por SQL, agora sem linhas duplicadas
df_logs.createOrReplaceTempView('logs_streaming_sem_duplicatas')

In [0]:
%sql
-- procura por valores duplicados nas colunas de IDs
select count(distinct session_id) as unique_session_id
    ,count(distinct user_id) as unique_user_id
    ,count(distinct movie_id) as unique_movie_id
    ,count(distinct session_id, user_id, movie_id) as row_id
    ,count(*) as total_linhas
from logs_streaming_sem_duplicatas

unique_session_id,unique_user_id,unique_movie_id,row_id,total_linhas
1001,80,60,997,1001


In [0]:
%sql
-- porque temos 997 combinacoes diferentes dos campos IDs, se ha 1001 linhas na base?
select *
from logs_streaming_sem_duplicatas
where session_id is null
    or user_id is null
    or movie_id is null

session_id,user_id,movie_id,device,country,subscription_type,watch_time_minutes,watch_date,rating,playback_status
SNULL001,null,M0001,null,Brazil,premium,null,31-31-2026,bad,error
SB76A14F1A2,null,M0047,null,Colômbia,standard,-15,05/05/2026,4,null
S5000FC182F,U0077,null,web,Chile,null,22,2026/01/03,4,buffering
S1B0D4AA15E,U0045,null,web,null,standard,45.5,03-12-2026,3,completed


## 2 Padronização dos dados

### 2.1 Ajustando o tipo de `watch_time_minutes`

Note que para a coluna `whatch_time_minutes` foi definido o tipo `string`, mas por se tratar de "quantidade de tempo em minutos", faria mais sentido se fosse fo tipo "número". Mas antes, vamos consultar a tabela temporária com SQL:

In [0]:
%sql 
-- verifica os valores unicos e ordena por ordem decrescente para ver se tem letras na coluna
select watch_time_minutes, count(*)
from logs_streaming_sem_duplicatas
group by watch_time_minutes
order by watch_time_minutes desc
limit 10

watch_time_minutes,count(*)
zero,14
9999,15
99,5
98,6
97,6
96,6
95,5
94,3
93,6
92,4


In [0]:
# seleciona valores unicos na coluna watch_time_minutes
minutos_unicos = df_logs.select('watch_time_minutes').distinct()

print("qtd valores unicos (minutos):", minutos_unicos.count())

minutos_unicos.sort(df_logs.watch_time_minutes.desc()).limit(10).display()

qtd valores unicos (minutos): 191


watch_time_minutes
zero
9999
99
98
97
96
95
94
93
92


In [0]:
from pyspark.sql.functions import (
    col, 
    expr, 
    when,
    coalesce, 
    lit # cria valor literal ou constante para todas as linhas de um dataframe
)

df_logs = df_logs.withColumn(
    'watch_time_minutes', 
    # col('watch_time_minutes').cast('float').cast('int') # falha por conta das linha com valor literal 'zero'
    expr('try_cast(watch_time_minutes as float)').cast('int')
).withColumn(
    'watch_time_minutes',
    when(
        col("watch_time_minutes") > 0, 
        col("watch_time_minutes")
    ).otherwise(
        0
    )
)

# # se ja quisesse tratar valores nulos nesse caso
# df_logs = df_logs.withColumn(
#     'watch_time_minutes', 
#     coalesce(
#         expr('try_cast(watch_time_minutes as float)').cast('int'), 
#         lit(0)
#     ) # try_cast retorna nulo quando nao eh possivel fazer cast de algum e coalesce coloca zero
# )

df_logs.limit(10).display()

session_id,user_id,movie_id,device,country,subscription_type,watch_time_minutes,watch_date,rating,playback_status
SD9E82F7AD2,U0021,M0010,Smart TV,Espanha,trial,166,2026-04-23,1,buffering
S5A91FD1643,U0026,M0016,null,Chile,basic,120,2026-05-07 00:23:00,0,erro
S04667B5896,U0058,M0006,mobile,México,premium,97,18/05/2026,3,cancelled
S5429269E17,U0010,M0018,smart_tv,México,standard,81,2026-05-20,4,paused
S6CF9145857,U0073,M0054,mobile,Espanha,free,36,2026-04-10 21:57:00,4,paused
S59BEFA0FF3,U0075,M0006,tablet,Colombia,BASIC,0,2026-03-17,1,completed
S002139AE2E,U0039,M0016,null,United States,standard,51,04-29-2026,0,completed
SD8C07AE960,U0041,M0007,null,BR,BASIC,20,01-19-2026,5,erro
S36BE651761,U0050,M0027,null,Portugal,basic,61,05-18-2026,1,null
S05C8E99E6A,U0019,M0017,null,Spain,Premium,0,2026-03-23,1,null


In [0]:
df_logs.printSchema()

root
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- movie_id: string (nullable = true)
 |-- device: string (nullable = true)
 |-- country: string (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- watch_time_minutes: integer (nullable = true)
 |-- watch_date: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- playback_status: string (nullable = true)



### 2.2 Criando novas colunas a partir de `watch_time_minutes`: tempo do filme em horas e classificação da duração do filme

In [0]:
# cria coluna watch_time_hours: tempo do filme em horas
from pyspark.sql.functions import col, round

df_logs = df_logs.withColumn(
    "watch_time_hours", # cria nova coluna chamada `watch_time_hours`
    round(col("watch_time_minutes")/60, 2) # valor da nova coluna eh o valor da coluna `watch_time_minutes` dividido por 60
)

df_logs.select('watch_time_minutes', 'watch_time_hours').limit(10).display()

watch_time_minutes,watch_time_hours
166,2.77
120,2.0
97,1.62
81,1.35
36,0.6
0,0.0
51,0.85
20,0.33
61,1.02
0,0.0


In [0]:
# cria coluna watch_category: categoriza filmes por tempo de visualizacao
from pyspark.sql.functions import when, col # nao ha necessidade de ficar reimportando, mas vou fazer isso nesse notebook

df_logs = df_logs.withColumn(
    "watch_category",
    when(
        (col("watch_time_minutes") >= 120) & (df_logs.watch_time_minutes.isNotNull()), 
        "Longo"
    ).otherwise(
        when(
            col("watch_time_minutes") < 120, 
            "Curto"
            ).otherwise(None)
    )
)

df_logs.select('watch_time_minutes', 'watch_time_hours', 'watch_category').limit(10).display()

watch_time_minutes,watch_time_hours,watch_category
166,2.77,Longo
120,2.0,Longo
97,1.62,Curto
81,1.35,Curto
36,0.6,Curto
0,0.0,Curto
51,0.85,Curto
20,0.33,Curto
61,1.02,Curto
0,0.0,Curto


In [0]:
df_logs.printSchema()

root
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- movie_id: string (nullable = true)
 |-- device: string (nullable = true)
 |-- country: string (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- watch_time_minutes: integer (nullable = true)
 |-- watch_date: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- playback_status: string (nullable = true)
 |-- watch_time_hours: double (nullable = true)
 |-- watch_category: string (nullable = true)



### 2.3 Padronização de colunas

#### 2.3.1 `playback_status`

Agora vamos passar para outra coluna: `playback_status`. Note que nessa coluna temos diferentes valores para o mesmo status, então devemos padronizar esses valores.

In [0]:
%sql
-- verificando a quantidade de dados por playback_status
select playback_status, count(*)
from logs_streaming
group by 1
order by 1 desc

playback_status,count(*)
paused,132
error,122
erro,107
completed,114
complete,100
cancelled,119
buffering,122
COMPLETED,100
null,91


In [0]:
# valores unicos da coluna playback_status
status_unicos = df_logs.select('playback_status').distinct()

print("qtd valores unicos (playback_status):", status_unicos.count())

status_unicos.sort('playback_status').display()

qtd valores unicos (playback_status): 9


playback_status
null
COMPLETED
buffering
cancelled
complete
completed
erro
error
paused


In [0]:
# passa todas as linhas da coluna playback_status para lower, sem substituir no dataframe
from pyspark.sql.functions import lower

(
    df_logs.withColumn(
        "playback_status",
        lower("playback_status")
    )
    .select('playback_status')
    .distinct()
    .display()
)

playback_status
buffering
erro
cancelled
paused
completed
null
error
complete


In [0]:
from pyspark.sql.functions import col, lower, trim, when, lit, create_map
from itertools import chain

# dicionario de mapeamento: valor original -> valor padronizado
mapeamento_status = {
    "complete": "completed",
    "erro": "error"
}

# Cria uma coluna do tipo map para fazer lookup no Spark
mapa_expr = create_map([lit(x) for x in chain(*mapeamento_status.items())])

df_logs = (
    df_logs
    .withColumn(
        "playback_status",
        lower(trim(col("playback_status")))  # normaliza: minusculo e sem espacos
    )
    .withColumn(
        "playback_status",
        when(col("playback_status").isin(list(mapeamento_status.keys())), mapa_expr[col("playback_status")])
        .otherwise(col("playback_status"))  # se nao esta no dicionario, mantem o valor original
    )
)

status_unicos = df_logs.select('playback_status').distinct()
print("qtd valores unicos (playback_status):", status_unicos.count())

df_logs.select("playback_status").distinct().display()

qtd valores unicos (playback_status): 6


playback_status
buffering
error
cancelled
paused
completed
null


#### 2.3.2 `subscription_type`

In [0]:
# verifica valores unicos de subscription_type
assinaturas_unicas = df_logs.select('subscription_type').distinct()

print("qtd valores unicos (subscription_type):", assinaturas_unicas.count())

assinaturas_unicas.sort('subscription_type').display()

qtd valores unicos (subscription_type): 8


subscription_type
null
BASIC
Premium
basic
free
premium
standard
trial


In [0]:
# passa todas as linhas de subscription_type para minuscula
from pyspark.sql.functions import lower

df_logs = df_logs.withColumn( "subscription_type", lower("subscription_type") )

df_logs.select('subscription_type').distinct().display()

subscription_type
trial
basic
premium
standard
free
null


#### 2.3.3 `country`

In [0]:
# verifica valores unicos de country
paises_unicos = df_logs.select('country').distinct()

print("qtd valores unicos (country):", paises_unicos.count())

paises_unicos.sort('country').display()

qtd valores unicos (country): 15


country
null
Argentina
BR
Brasil
Brazil
Chile
Colombia
Colômbia
Espanha
Mexico


In [0]:
# padroniza valores da coluna country
from pyspark.sql.functions import col, lower, trim, when, lit, create_map
from itertools import chain

# dicionario de mapeamento para country: valor original -> valor padronizado
mapeamento_status = {
    "brazil": "brasil",
    "br": "brasil",
    "colômbia": "colombia",
    "spain": "espanha",
    "méxico": "mexico",
    "united states": "usa"
}

# Cria uma coluna do tipo map para fazer lookup no Spark
mapa_expr = create_map([lit(x) for x in chain(*mapeamento_status.items())])

df_logs = (
    df_logs
    .withColumn(
        "country",
        lower(trim(col("country")))  # normaliza: minusculo e sem espacos
    )
    .withColumn(
        "country",
        when(col("country").isin(list(mapeamento_status.keys())), mapa_expr[col("country")])
        .otherwise(col("country"))  # se nao esta no dicionario, mantem o valor original
    )
)

status_unicos = df_logs.select('country').distinct()
print("qtd valores unicos (country):", status_unicos.count())

df_logs.select("country").distinct().display()

qtd valores unicos (country): 9


country
espanha
chile
mexico
colombia
usa
brasil
portugal
null
argentina


#### 2.3.4 `device`

In [0]:
# valores unicos da coluna device
status_unicos = df_logs.select('device').distinct()

print("qtd valores unicos (device):", status_unicos.count())

status_unicos.sort('device').display()

qtd valores unicos (device): 8


device
null
MOBILE
Smart TV
console
mobile
smart_tv
tablet
web


In [0]:
# padroniza valores da coluna device
from pyspark.sql.functions import col, lower, trim, when, lit, create_map, regexp_replace
from itertools import chain

# dicionario de mapeamento: valor original -> valor padronizado
mapeamento_status = {
    "complete": "completed",
    "erro": "error"
}

# Cria uma coluna do tipo map para fazer lookup no Spark
mapa_expr = create_map([lit(x) for x in chain(*mapeamento_status.items())])

df_logs = (
    df_logs
    .withColumn(
        "device",
        lower(trim(col("device")))  # normaliza: minúsculo e sem espaços
    )
    .withColumn('device', regexp_replace('device', '_', ' '))
)

status_unicos = df_logs.select('device').distinct()
print("qtd valores unicos (device):", status_unicos.count())

df_logs.select("device").distinct().display()

qtd valores unicos (device): 6


device
smart tv
null
mobile
tablet
console
web


#### 2.3.5 `rating`

In [0]:
%sql select rating, count(*) from logs_streaming group by rating order by 1 desc

rating,count(*)
bad,14
6,15
5,187
4.5,20
4,196
3,161
2,181
1,177
0,15
null,41


In [0]:
df_logs = df_logs

# padroniza valores da coluna rating
from pyspark.sql.functions import col, lower, trim, when, lit, create_map, regexp_replace
from itertools import chain

df_logs = (
    df_logs
    .withColumn(
        'rating', 
        expr('try_cast(rating as float)').cast('int')
    )
    .withColumn(
        'rating',
        when(col("rating") > 5, 5).otherwise(col("rating"))
    )
)

ratings_unicos = df_logs.select('rating').distinct()
print("qtd valores unicos (rating):", ratings_unicos.count())

df_logs.select("rating").distinct().sort('rating').display()

qtd valores unicos (rating): 7


rating
null
0
1
2
3
4
5


### 2.4 Ajustando o formato e tipo de `watch_date`

In [0]:
# cria df apenas com datas
df_dates = df_logs.select(col("watch_date").alias("wd_string"))

df_dates.display()

wd_string
2026-04-23
2026-05-07 00:23:00
18/05/2026
2026-05-20
2026-04-10 21:57:00
2026-03-17
04-29-2026
01-19-2026
05-18-2026
2026-03-23


In [0]:
from pyspark.sql.functions import col, coalesce, try_to_date, split

# funcao para converter multiplos formatos de data
def to_date_multi_format(coluna, 
                         formatos_suportados=("MM/dd/yyyy", "yyyy-MM-dd", "dd/MM/yyyy", "yyyy/MM/dd", "MM-dd-yyyy", "yyyy-MM-dd HH:MM:SS")
):
    return coalesce(*[try_to_date(coluna, f) for f in formatos_suportados])

# teste da funcao
df_dates = df_dates.withColumn(
    "wd_date", to_date_multi_format(col("wd_string"))
).withColumn(
    "wd_string_split", split(df_dates["wd_string"], ' ')[0] # quebra string em array e pega o primeiro elemento, que eh a data do dia
).withColumn(
    "wd_string_split_date", to_date_multi_format(col("wd_string_split")) # aplica funcao na string da data
).withColumn(
    "wd_string_split_date_direto", to_date_multi_format(split(df_dates["wd_string"], ' ')[0]) # aplica funcao direto no resultado do split
)

df_dates.filter( col("wd_date").isNull() ).display()


wd_string,wd_date,wd_string_split,wd_string_split_date,wd_string_split_date_direto
2026-05-07 00:23:00,null,2026-05-07,2026-05-07,2026-05-07
2026-04-10 21:57:00,null,2026-04-10,2026-04-10,2026-04-10
2026-01-28 05:41:00,null,2026-01-28,2026-01-28,2026-01-28
2026-03-12 23:16:00,null,2026-03-12,2026-03-12,2026-03-12
2026-03-21 14:31:00,null,2026-03-21,2026-03-21,2026-03-21
2026-02-19 19:34:00,null,2026-02-19,2026-02-19,2026-02-19
2026-01-31 05:55:00,null,2026-01-31,2026-01-31,2026-01-31
2026-03-03 05:32:00,null,2026-03-03,2026-03-03,2026-03-03
2026-03-21 10:27:00,null,2026-03-21,2026-03-21,2026-03-21
2026-04-29 14:15:00,null,2026-04-29,2026-04-29,2026-04-29


In [0]:
df_logs = df_logs.withColumn(
    "watch_date", 
    to_date_multi_format(split(df_logs["watch_date"], ' ')[0])
)

df_logs.select("watch_date").display()


watch_date
2026-04-23
2026-05-07
2026-05-18
2026-05-20
2026-04-10
2026-03-17
2026-04-29
2026-01-19
2026-05-18
2026-03-23


In [0]:
df_logs.filter( col("watch_date").isNull() ).display()

session_id,user_id,movie_id,device,country,subscription_type,watch_time_minutes,watch_date,rating,playback_status,watch_time_hours,watch_category
SNULL001,null,M0001,null,brasil,premium,0,null,null,error,0.0,Curto


In [0]:
df_logs.printSchema()

root
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- movie_id: string (nullable = true)
 |-- device: string (nullable = true)
 |-- country: string (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- watch_time_minutes: integer (nullable = true)
 |-- watch_date: date (nullable = true)
 |-- rating: integer (nullable = true)
 |-- playback_status: string (nullable = true)
 |-- watch_time_hours: double (nullable = true)
 |-- watch_category: string (nullable = true)



### 2.5 Lidando com valores nulos

Até vimos que existem valores nulos em praticamente todas as colunas. Em algumas delas a falta de valor não necessáriamente é um problema que impacta na análise, enquanto que em outras, a falta desse valor é um pouco mais problemática. 

Por exemplo, no caso de `user_id`, não faz sentido manter essa linha na base de logs, pois não será possível relacioná-la com a base de usuários. Já para a coluna `country`, podemos apenas preencher as linhas vazias com um valor "coringa", pois a falta dessa informação não trará muitos prejuízos.

Começo criando uma nova tabela temporária para verificar via SQL a quantidade de linhas nulas em cada coluna:

In [0]:
# cria tabela temporaria para ser acessada por SQL para investigar valores nulos
df_logs.createOrReplaceTempView('logs_streaming_nulos')

In [0]:
%sql
select count(*) as total_linhas
    ,count_if(session_id is null) as qtd_nulos_user_id
    ,count_if(user_id is null) as qtd_nulos_user_id
    ,count_if(movie_id is null) as qtd_nulos_movie_id
    ,count_if(device is null) as qtd_nulos_device
    ,count_if(country is null) as qtd_nulos_country
    ,count_if(subscription_type is null) as qtd_nulos_subscription_type
    ,count_if(watch_time_minutes is null) as qtd_nulos_watch_time_minutes
    ,count_if(watch_date is null) as qtd_nulos_watch_date
    ,count_if(rating is null) as qtd_nulos_rating
    ,count_if(playback_status is null) as qtd_nulos_playback_status
    ,count_if(watch_time_hours is null) as qtd_nulos_watch_time_hours
    ,count_if(watch_category is null) as qtd_nulos_watch_category
from logs_streaming_nulos

total_linhas,qtd_nulos_user_id,qtd_nulos_user_id,qtd_nulos_movie_id,qtd_nulos_device,qtd_nulos_country,qtd_nulos_subscription_type,qtd_nulos_watch_time_minutes,qtd_nulos_watch_date,qtd_nulos_rating,qtd_nulos_playback_status,qtd_nulos_watch_time_hours,qtd_nulos_watch_category
1001,0,2,2,255,137,127,0,1,54,90,0,0


In [0]:
# essas colunas sao imprescindiveis que tenham valor
df_logs = df_logs.dropna(subset=['user_id']).dropna(subset=['movie_id'])

In [0]:
# verifica valores unicos de country
paises_unicos = df_logs.select('country').distinct()

print("qtd valores unicos (country):", paises_unicos.count())

paises_unicos.sort('country').display()

qtd valores unicos (country): 9


country
null
argentina
brasil
chile
colombia
espanha
mexico
portugal
usa


In [0]:
from pyspark.sql.functions import col

df_nulos_country = df_logs.filter( col("country").isNull() )

print(df_nulos_country.count())
df_nulos_country.limit(10).display()

136


session_id,user_id,movie_id,device,country,subscription_type,watch_time_minutes,watch_date,rating,playback_status,watch_time_hours,watch_category
S8521C6FE8D,U0029,M0023,tablet,null,standard,12,2026-03-16,4,paused,0.2,Curto
SC8E7DE44F0,U0078,M0037,null,null,null,173,2026-05-18,4,cancelled,2.88,Longo
S0EBF011691,U0016,M0014,null,null,basic,2,2026-01-31,3,error,0.03,Curto
S4B3D2E05E1,U0020,M0044,console,null,basic,27,2026-03-19,1,completed,0.45,Curto
S0D5C01CE6A,U0018,M0024,null,null,premium,175,2026-04-15,5,error,2.92,Longo
S63A3575BA4,U0057,M0057,smart tv,null,premium,20,2026-04-15,4,error,0.33,Curto
S219F96138C,U0076,M0013,console,null,premium,78,2026-03-21,null,error,1.3,Curto
SE82DEA92B4,U0009,M0014,tablet,null,basic,62,2026-04-29,3,buffering,1.03,Curto
S944D2D4A5C,U0069,M0049,smart tv,null,premium,169,2026-03-03,3,paused,2.82,Longo
SF25381EC09,U0060,M0026,web,null,trial,158,2026-02-17,2,error,2.63,Longo


In [0]:
# conta a quantidade de nulos em country
from pyspark.sql.functions import count, when, col

df_logs.select(
    count(
        when(col("country").isNull(), 1)
    ).alias("total_nulos_country")
).display()

total_nulos_country
136


In [0]:
# preenche valores nulos de cada coluna com valor especifico
df_logs = df_logs.fillna({
    "country": "desconhecido",
    "subscription_type": "unknown"
})

df_logs.limit(10).display()

session_id,user_id,movie_id,device,country,subscription_type,watch_time_minutes,watch_date,rating,playback_status,watch_time_hours,watch_category
SD9E82F7AD2,U0021,M0010,smart tv,espanha,trial,166,2026-04-23,1,buffering,2.77,Longo
S5A91FD1643,U0026,M0016,null,chile,basic,120,2026-05-07,0,error,2.0,Longo
S04667B5896,U0058,M0006,mobile,mexico,premium,97,2026-05-18,3,cancelled,1.62,Curto
S5429269E17,U0010,M0018,smart tv,mexico,standard,81,2026-05-20,4,paused,1.35,Curto
S6CF9145857,U0073,M0054,mobile,espanha,free,36,2026-04-10,4,paused,0.6,Curto
S59BEFA0FF3,U0075,M0006,tablet,colombia,basic,0,2026-03-17,1,completed,0.0,Curto
S002139AE2E,U0039,M0016,null,usa,standard,51,2026-04-29,0,completed,0.85,Curto
SD8C07AE960,U0041,M0007,null,brasil,basic,20,2026-01-19,5,error,0.33,Curto
S36BE651761,U0050,M0027,null,portugal,basic,61,2026-05-18,1,null,1.02,Curto
S05C8E99E6A,U0019,M0017,null,espanha,premium,0,2026-03-23,1,null,0.0,Curto


In [0]:
# preenche valores nulos de todas colunas do dataframe
df_logs = df_logs.fillna("sem informacao")

df_logs.limit(10).display()

session_id,user_id,movie_id,device,country,subscription_type,watch_time_minutes,watch_date,rating,playback_status,watch_time_hours,watch_category
SD9E82F7AD2,U0021,M0010,smart tv,espanha,trial,166,2026-04-23,1,buffering,2.77,Longo
S5A91FD1643,U0026,M0016,sem informacao,chile,basic,120,2026-05-07,0,error,2.0,Longo
S04667B5896,U0058,M0006,mobile,mexico,premium,97,2026-05-18,3,cancelled,1.62,Curto
S5429269E17,U0010,M0018,smart tv,mexico,standard,81,2026-05-20,4,paused,1.35,Curto
S6CF9145857,U0073,M0054,mobile,espanha,free,36,2026-04-10,4,paused,0.6,Curto
S59BEFA0FF3,U0075,M0006,tablet,colombia,basic,0,2026-03-17,1,completed,0.0,Curto
S002139AE2E,U0039,M0016,sem informacao,usa,standard,51,2026-04-29,0,completed,0.85,Curto
SD8C07AE960,U0041,M0007,sem informacao,brasil,basic,20,2026-01-19,5,error,0.33,Curto
S36BE651761,U0050,M0027,sem informacao,portugal,basic,61,2026-05-18,1,sem informacao,1.02,Curto
S05C8E99E6A,U0019,M0017,sem informacao,espanha,premium,0,2026-03-23,1,sem informacao,0.0,Curto


In [0]:
# cria tabela temporaria, agora ja sem nulos
df_logs.createOrReplaceTempView('logs_streaming_sem_nulos')

In [0]:
%sql
select count_if(device is null) as qtd_nulos_device
    ,count_if(country is null) as qtd_nulos_country
    ,count_if(subscription_type is null) as qtd_nulos_subscription_type
    ,count_if(watch_time_minutes is null) as qtd_nulos_watch_time_minutes
    ,count_if(watch_date is null) as qtd_nulos_watch_date
    ,count_if(rating is null) as qtd_nulos_rating
    ,count_if(playback_status is null) as qtd_nulos_playback_status
    ,count_if(watch_time_hours is null) as qtd_nulos_watch_time_hours
    ,count_if(watch_category is null) as qtd_nulos_watch_category
from logs_streaming_sem_nulos

qtd_nulos_device,qtd_nulos_country,qtd_nulos_subscription_type,qtd_nulos_watch_time_minutes,qtd_nulos_watch_date,qtd_nulos_rating,qtd_nulos_playback_status,qtd_nulos_watch_time_hours,qtd_nulos_watch_category
0,0,0,0,0,53,0,0,0


## 3 Filtros e agregações

### 3.1 Filtrando sessões completas

In [0]:
sessoes_finalizadas = (
    df_logs.filter(
        (col('playback_status') == "completed") # seleciono sessoes finalizadas
        & (col('watch_time_minutes') > 0) # AND com duracao maior que zero
    )
)

print("qtd sessoes finalizadas:", sessoes_finalizadas.count())

sessoes_finalizadas.limit(10).display()

qtd sessoes finalizadas: 293


session_id,user_id,movie_id,device,country,subscription_type,watch_time_minutes,watch_date,rating,playback_status,watch_time_hours,watch_category
S002139AE2E,U0039,M0016,sem informacao,usa,standard,51,2026-04-29,0,completed,0.85,Curto
SB2ED3FC1E9,U0061,M0017,smart tv,usa,premium,57,2026-02-16,3,completed,0.95,Curto
S57D9A825D4,U0076,M0008,console,brasil,free,164,2026-04-04,1,completed,2.73,Longo
S19DF76D96C,U0035,M0032,web,colombia,standard,73,2026-08-04,2,completed,1.22,Curto
S4B3D2E05E1,U0020,M0044,console,desconhecido,basic,27,2026-03-19,1,completed,0.45,Curto
SA3EEFA132D,U0017,M0033,smart tv,brasil,premium,68,2026-03-22,5,completed,1.13,Curto
SC379A83B2C,U0003,M0053,console,portugal,basic,123,2026-03-03,4,completed,2.05,Longo
S70C514CED4,U0062,M0018,smart tv,colombia,unknown,167,2026-01-11,2,completed,2.78,Longo
SAE294B9070,U0045,M0046,web,portugal,premium,134,2026-05-05,3,completed,2.23,Longo
SBE73DC86EB,U0048,M0031,mobile,colombia,trial,130,2026-04-13,5,completed,2.17,Longo


### 3.2 Agragando por `subscription_type`

In [0]:
from pyspark.sql.functions import count, avg, sum

(
    df_logs.groupBy("subscription_type")
    .agg(
        count("*").alias("total_sessoes"),
        avg("watch_time_minutes").alias("media_minutos_assistidos"),
        sum("watch_time_minutes").alias("total_minutos_assistidos")
    )
    # .orderBy("total_sessoes", ascending=False)
    .orderBy(col("total_sessoes").desc())
).display()

subscription_type,total_sessoes,media_minutos_assistidos,total_minutos_assistidos
basic,254,200.33858267716536,50886
premium,254,248.28346456692913,63064
standard,130,170.2923076923077,22138
unknown,126,244.9126984126984,30859
trial,122,328.6885245901639,40100
free,111,256.27027027027026,28446


In [0]:
%sql
-- note que o resultado da agregacao feita na tabela temporaria do SQL eh diferente, porque ela foi definida antes das transformacoes
select subscription_type, count(*) 
from logs_streaming
group by 1 
order by 1 desc

subscription_type,count(*)
trial,122
standard,133
premium,125
free,111
basic,126
Premium,135
BASIC,128
null,127


### 3.3 Agregando por país

In [0]:
df_logs.groupBy("country")\
    .agg(
        count("*").alias("total_sessoes"),
        avg("watch_time_minutes").alias("media_minutos_assistidos"),
    )\
    .orderBy("total_sessoes", ascending=False)\
    .display()

country,total_sessoes,media_minutos_assistidos
brasil,170,260.9823529411765
mexico,137,304.2043795620438
desconhecido,136,81.05147058823529
colombia,123,97.34146341463415
usa,117,417.8974358974359
espanha,103,276.59223300970876
chile,76,92.34210526315789
argentina,72,219.52777777777777
portugal,63,416.6190476190476


## 4 Operações de ajuste e limpeza no arquivo de logs

In [0]:
from pyspark.sql.functions import col, expr, when, round, create_map, lit, lower, trim, regexp_replace, coalesce, try_to_date, split
from itertools import chain

df_logs = spark.read.csv("/Volumes/academy/dados-streaing-filmes/logs_streaming.csv", header=True, inferSchema=True)
print((df_logs.count(), len(df_logs.columns)))

df_logs = df_logs.dropDuplicates(subset=['session_id', 'user_id', 'movie_id'])

df_logs = (df_logs
           .withColumn('watch_time_minutes', expr('try_cast(watch_time_minutes as float)').cast('int'))
           .withColumn('watch_time_minutes', when(col("watch_time_minutes") > 0, col("watch_time_minutes")).otherwise(0)))

df_logs = df_logs.withColumn("watch_time_hours", round(col("watch_time_minutes")/60, 2))

df_logs = (df_logs
           .withColumn("watch_category", when((col("watch_time_minutes") >= 120) & (df_logs.watch_time_minutes.isNotNull()), "Longo")
                       .otherwise(when(col("watch_time_minutes") < 120, "Curto").otherwise(None))))

mapeamento_status = {"complete": "completed", "erro": "error"}
mapa_expr_status  = create_map([lit(x) for x in chain(*mapeamento_status.items())])
df_logs           = (df_logs
                     .withColumn("playback_status", lower(trim(col("playback_status"))))
                     .withColumn("playback_status", when(col("playback_status").isin(list(mapeamento_status.keys())), mapa_expr_status[col("playback_status")])
                                 .otherwise(col("playback_status"))))

df_logs = df_logs.withColumn( "subscription_type", lower("subscription_type") )

mapeamento_country = {"brazil": "brasil", "br": "brasil", "colômbia": "colombia", "spain": "espanha", "méxico": "mexico", "united states": "usa"}
mapa_expr_country  = create_map([lit(x) for x in chain(*mapeamento_country.items())])
df_logs = (df_logs
           .withColumn("country", lower(trim(col("country"))))
           .withColumn("country", when(col("country").isin(list(mapeamento_country.keys())), mapa_expr_country[col("country")]).otherwise(col("country"))))

df_logs = (df_logs
           .withColumn("device", lower(trim(col("device"))))
           .withColumn('device', regexp_replace('device', '_', ' ')))

df_logs = (df_logs
           .withColumn('rating', expr('try_cast(rating as float)').cast('int'))
           .withColumn('rating', when(col("rating") > 5, 5).otherwise(col("rating"))))

def to_date_multi_format(coluna, formatos_suportados=("MM/dd/yyyy", "yyyy-MM-dd", "dd/MM/yyyy", "yyyy/MM/dd", "MM-dd-yyyy", "yyyy-MM-dd HH:MM:SS")):
    return coalesce(*[try_to_date(coluna, f) for f in formatos_suportados])
df_logs = df_logs.withColumn("watch_date", to_date_multi_format(split(df_logs["watch_date"], ' ')[0]))

df_logs = df_logs.dropna(subset=['user_id']).dropna(subset=['movie_id']).dropna(subset=['session_id'])

df_logs = df_logs.fillna({"country": "desconhecido", "subscription_type": "unknown"})

df_logs = df_logs.fillna("sem informacao")

print((df_logs.count(), len(df_logs.columns)))

(1007, 10)
(997, 12)


In [0]:
df_logs.printSchema()

root
 |-- session_id: string (nullable = false)
 |-- user_id: string (nullable = false)
 |-- movie_id: string (nullable = false)
 |-- device: string (nullable = false)
 |-- country: string (nullable = false)
 |-- subscription_type: string (nullable = false)
 |-- watch_time_minutes: integer (nullable = true)
 |-- watch_date: date (nullable = true)
 |-- rating: integer (nullable = true)
 |-- playback_status: string (nullable = false)
 |-- watch_time_hours: double (nullable = true)
 |-- watch_category: string (nullable = false)

